# H1. Learning a probability
Book: Logistic Regression, Model Fitting and Confusion Matrix.

Supplement: Alisa's book of LLMs, [Information theory](https://alisawuffles.notion.site/alisa-s-book-of-llms#2e97eb8736058077b09dea853f62c399) and [Numerical stability](https://alisawuffles.notion.site/alisa-s-book-of-llms#3157eb87360580ec9b01c45ffec58c03). Read the cross-entropy and log-sum-exp portions.
These selected readings are optional support. The classroom examples define the required scope.
Reading caution: generic PyTorch cross-entropy does not shift token labels.
A model-specific language-model wrapper may align them separately.

Circle data follow the book's example, scaled by 10. Inner=0, outer=1.
The fixed split will recur in H4 and H5. Keep the test observations unused.

## Setup
Run this cell once. Helpers support the experiments below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize': (8, 4.5), 'font.size': 12})

def circle_data(seed=601, n=200):
    rng = np.random.default_rng(seed)
    n0 = n // 2
    radius = np.r_[rng.uniform(0, .4, n0), rng.uniform(.8, 1, n-n0)]
    angle = rng.uniform(0, 2*np.pi, n)
    x = np.c_[radius*np.cos(angle), radius*np.sin(angle)]
    y = np.r_[np.zeros(n0), np.ones(n-n0)].astype(int)
    return x, y

def circle_split():
    from sklearn.model_selection import train_test_split
    x, y = circle_data()
    xa, xt, ya, yt = train_test_split(x, y, test_size=.2, stratify=y,
                                     random_state=600)
    xr, xv, yr, yv = train_test_split(xa, ya, test_size=.25, stratify=ya,
                                     random_state=600)
    return xr, xv, xt, yr, yv, yt

def radial_features(x):
    return np.sum(x*x, axis=1, keepdims=True)

def logistic_model(radial=False):
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler, FunctionTransformer
    from sklearn.linear_model import LogisticRegression
    steps = [FunctionTransformer(radial_features)] if radial else []
    return make_pipeline(*steps, StandardScaler(), LogisticRegression(C=1.0))

def classifier_plot(probability, x, y, ax=None, title=""):
    if ax is None:
        _, ax = plt.subplots()
    g = np.linspace(-1.1, 1.1, 130)
    xx, yy = np.meshgrid(g, g)
    p = probability(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, p, levels=np.linspace(0, 1, 11), cmap="RdBu_r",
                alpha=.35, vmin=0, vmax=1)
    if p.min() < .5 < p.max():
        ax.contour(xx, yy, p, levels=[.5], colors="black", linewidths=1.5)
    ax.scatter(x[:, 0], x[:, 1], c=y, cmap="RdBu_r", vmin=0, vmax=1,
               edgecolors="white", s=25)
    ax.set(xlabel="x1", ylabel="x2", title=title, aspect="equal")
    return ax

## A. Logit, probability, boundary (30 minutes)
Predict how increasing the intercept shifts the boundary.
Change just one coefficient, then explain the change in probabilities.

In [ ]:
from scipy.special import expit
xr, xv, xt, yr, yv, yt = circle_split()
intercept, beta1, beta2 = 0., 2., -1.
probability = lambda x: expit(intercept + beta1*x[:, 0] + beta2*x[:, 1])
classifier_plot(probability, xr, yr, title="A chosen logistic predictor")
plt.show()

In [ ]:
model = logistic_model().fit(xr, yr)
classifier_plot(lambda x: model.predict_proba(x)[:, 1], xr, yr,
                title="A fitted linear logit")
plt.show()

Why does changing a threshold fail to create a circular boundary?
A prediction in [0,1] need not equal a calibrated probability.

Prediction:

Observation:

Explanation:

## B. A threshold is a decision (15 minutes)
Use the fixed probabilities below. They are invented for this accounting task.
Predict the effect of a higher false-negative cost before changing the threshold.

In [ ]:
probabilities = np.array([.10, .32, .35, .45, .55, .70, .85, .95])
outcomes = np.array([0, 0, 1, 0, 1, 1, 0, 1])
threshold = .5
false_positive_cost, false_negative_cost = 1, 1
actions = probabilities >= threshold
fp = np.sum(actions & (outcomes == 0))
fn = np.sum(~actions & (outcomes == 1))
print("False positives:", fp, "False negatives:", fn)
print("Realized cost:", false_positive_cost*fp + false_negative_cost*fn)
fig, ax = plt.subplots()
ax.scatter(np.arange(len(outcomes)), probabilities, c=outcomes, cmap="RdBu_r")
ax.axhline(threshold, color="black")
ax.set(xlabel="Case", ylabel="Predicted probability", ylim=(0, 1))
plt.show()

Try thresholds 0.3, 0.5, and 0.8. Then change false_negative_cost to 5 and repeat.
This tiny table illustrates the tradeoff,
not an estimate of deployment performance or a reliable optimized threshold.

Prediction:

Observation:

Explanation:

## C. Three classes and stable softmax (15 minutes)
Predict the probabilities for log(3), log(2), log(1).
Then add 1000 to every logit. Should the probabilities change?

In [ ]:
from scipy.special import logsumexp
logits = np.log([3., 2., 1.])
def stable_softmax(scores):
    shifted = scores-np.max(scores)
    return np.exp(shifted)/np.exp(shifted).sum()
class_probability = stable_softmax(logits)
target_class = 0
cross_entropy = logsumexp(logits)-logits[target_class]
print("Probabilities:", class_probability, "loss:", cross_entropy)
print("After common shift:", stable_softmax(logits+1000))
assert np.allclose(class_probability, [.5, 1/3, 1/6])
assert np.allclose(stable_softmax(logits+1000), class_probability)
assert np.isclose(cross_entropy, np.log(2))

Explain why shifting all scores does not change their relative evidence.
Why use log-softmax or a combined loss rather than log of an underflowed probability?

Prediction:

Observation:

Explanation:

## Individual exit
For y=0, which probability has a larger log loss: 0.6 or 0.99? Why?
What changes in the fitted model when only the decision threshold changes?